In [1]:
import sys
sys.path.append("/Users/sujaladhikari/Sujal's Personal/Projects/FedIDS")

In [2]:
import os 
import shutil
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torch 
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from Model.model import MLP
from torch.optim import Adam
import utils
from utils import JoinCustomDataset
from sklearn.metrics import classification_report
from federatedlearning import updatefrom_local, weight_averaging
from nids_training import evaluate_model
import matplotlib.pyplot as plt 
import random 

### Setting up the device

In [4]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device
RANDOMSEED = 42
random.seed(RANDOMSEED)
np.random.seed(RANDOMSEED)
torch.manual_seed(RANDOMSEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(RANDOMSEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
### Creating a generator to pass into the data loader 
g = torch.Generator()
g.manual_seed(RANDOMSEED)

### Creating the global model - using the same MLP used for the centralized model 

In [ ]:
input_size = 78
hidden_layer = [256, 128,64,8]
num_classes = 2
global_model = MLP(input_size, hidden_layer,num_classes).to(device)
global_model
num_clients = 4

### Creating two folders that holds the training input data, and the other holds the result 

In [ ]:
manual = ['One', 'Two', 'Three' , 'Four']
types = ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']
source_directory = '../silos_datasets/'
destination_directory  = './client_data/nids/' ## We are creating sub-directory in order to check on the future datasets.
os.makedirs(destination_directory , exist_ok=True)
for index,centralized_test_loss in enumerate(manual):
    new_client_name = f'client_{index}'
    for type in types:
        old_file = f'silo{centralized_test_loss}_{type}.csv'
        old_filepath = os.path.join(source_directory, old_file)
        
        new_file = f'{new_client_name}_{type}.csv'
        new_filepath = os.path.join(destination_directory , new_file)

        if os.path.exists(old_filepath):
            shutil.copy(old_filepath, new_filepath)

saving_directory = os.path.join('./output_nids/DNN_fedavg/nids') ## Creating the folder that stores the result of each client performance 
os.makedirs(saving_directory, exist_ok=True)

### Creating the Data Configuration and Training Configuration 


In [ ]:
batch_size = 64 ## Initially we set up as same as the centralized model 
lr = 0.0001 ## different learning rate
num_rounds = 5 ## 5/.0001 => 50000 rounds 
num_local_epochs = 5
save_interval = 1

### We will be creating new dataset which contains all the testing data, that will keep on checking on how the global model is performing !?

### This will be the golden dataset that will contain 50% of the randomly sampled data from each of the testing data of the clients, and to make it more further we will be shuffling the data and will take the 100% of the whole shuffle data !

In [ ]:
## Creating an ultimate gloden dataset 
golden_data = []
path = './client_data/nids/'
for i in range(num_clients):
    filepath = os.path.join(path, f'client_{i}_X_test.csv')
    testfilepath = os.path.join(path, f'client_{i}_y_test.csv')
    if os.path.exists(filepath) and os.path.exists(testfilepath):
        x_test_data = pd.read_csv(filepath)
        test_data = pd.read_csv(testfilepath)
        combined_data = pd.concat([x_test_data,test_data], axis = 1)
        golden_data.append(combined_data)


global_dataset = pd.concat(golden_data, ignore_index=True)
## Randomly shuffling the dataset 
global_dataset = global_dataset.sample(frac = 1, random_state = RANDOMSEED).reset_index(drop = True)
global_dataset.to_csv('../datasets/global_test_dataset.csv', index = False)
global_dataset.head(5)

### Two Global Metrics for the global model 

In [ ]:
performance_dict, performance_log = dict(), dict()
metric_keys = ['g_train_loss', 'g_test_loss']
performance_dict, performance_log = utils.performance_analyzer(metric_keys)

In [ ]:
labmda = 0.01

### Loading all the client data before training them to the global model 


In [ ]:
client_loaders = [] ## It has four dataloaders for each client 
for index in range(num_clients):
    features_path = f'client_{index}_X_train.csv'
    labels_path = f'client_{index}_y_train.csv'
    features_directory = os.path.join(destination_directory, features_path )
    labels_directory = os.path.join(destination_directory, labels_path) 
    dataset = utils.JoinCustomDataset(features_directory, labels_directory)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle = True,generator=g) ## The batch size is 64
    client_loaders.append(dataloader)


### Creating a test loader for each client to check the test loss 

In [ ]:
validation_loaders = []
for index in range(num_clients):
    features_path = f'client_{index}_X_val.csv'
    labels_path = f'client_{index}_y_val.csv'
    features_directory = os.path.join(destination_directory, features_path )
    labels_directory = os.path.join(destination_directory, labels_path)
    print(features_directory,labels_directory)
    dataset = utils.JoinCustomDataset(features_directory, labels_directory)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle = True,generator=g)
    validation_loaders.append(dataloader)

### Automatically resuming from the checkpoint 


In [ ]:
# Checking if there is already anything going on 
log_path = os.path.join(saving_directory, 'performanance_log.pickle')
if os.path.isfile(log_path):
    performance_log = utils.loading_pickle(log_path)
starting_round = len(performance_log[metric_keys[0]]) ## Check the list of the stored values (g_train), if the value is greaeter thean 0 then the model is already started and doing its job, and if the model crashes then it can continue from where it left!
if starting_round > 0:
    global_model.load_state_dict(torch.load(os.path.join(saving_directory, 'g_r_{}.pth').format(starting_round))) ## The global model takes the weight from where it left 

In [ ]:
global_weights = global_model.state_dict() ## This gives the initial weights of the given model
loss_function = nn.CrossEntropyLoss()
optimization_args = {'lr':lr}

### Starting of the model 

### We wll be looping the model from start round to number of rounds
### Here the starting round is initially zero as nothing is loaded in the given model 

for round_number in range(starting_round, num_rounds): ## Initially 0 -> 5
    print('Round Number:', round_number) 
    global_model.train()
    client_updates = dict() ## We will be storing the updates given by clients in a dictionary 
    for client_number in range(num_clients): ## Looping through each client
        print('Client:', client_number) ## Printing out the client number 
        client_loader = client_loaders[client_number] ## Loading the each client data 
        validation_loader = validation_loaders[client_number] ## Loading each validation data 
        client_update = updatefrom_local(global_model,client_loader, validation_loader, num_local_epochs, optimization_args) 
        client_updates.setdefault('local_weights', list()).append(client_update['local_weights'])
        client_updates.setdefault('num_samples', list()).append(client_update['num_samples'])

        ## Performance log update 
        performance_log.setdefault('c_{}_train_loss'.format(client_number), list()).append(client_update['train_loss'])
        ## Train loss of each client using the global model on training data 
        performance_log.setdefault('c_{}_test_loss'.format(client_number), list()).append(client_update['test_loss'])

    
    global_weights = weight_averaging(client_updates['local_weights'], client_updates['num_samples'], device)
    global_model.load_state_dict(global_weights) ## Updating the weights to the global model 

    ## Evaluating the global model on local dataset in each round 
    for client_index in range(num_clients):
        g_train_loss = evaluate_model(global_model, client_loaders[client_index], loss_function, tqdm_desc = 'g_train_loss')
        performance_dict['g_train_loss'].update_state(g_train_loss)
        g_test_loss = evaluate_model(global_model, validation_loaders[client_index], loss_function, tqdm_desc='Validation Loss' )
        performance_dict['g_test_loss'].update_state(g_test_loss)
    
    performance_log['g_train_loss'].append(performance_dict['g_train_loss'].result())
    performance_log['g_test_loss'].append(performance_dict['g_test_loss'].result())
    performance_dict['g_train_loss'].reset_state()
    performance_dict['g_test_loss'].reset_state()

    for metric in metric_keys:
        print(f"{metric}: {performance_log[metric][-1]}")

    ## Saving the global model 

    if (round_number + 1)  % save_interval == 0: 
        torch.save(global_model.state_dict(), os.path.join(saving_directory, 'g_r_{}.pth'.format(round_number+1))) ## Saving the global model's weights in the given directory with the name g_r_1..n.pth
        utils.savein_pickle(log_path,performance_log)  ## Storing the overall value in the pickle form to access it later 
    

### The global weight is updated in each round where the weight of each client for each layer is stored in the client updates which is then sent to weight_averaging which averages the weight based on the samples size


### Making a dataframe that stores all the test loss and train loss of each client and global model in each round



In [ ]:
import pickle
with open('./output_nids/DNN_fedavg/nids/performanance_log.pickle', 'rb') as f:
    losses = pickle.load(f)

data = pd.DataFrame(losses)
data.index.name = 'Round'
data = data.reset_index()
data.head(5)

In [ ]:
plt.plot(data['g_train_loss'], label = 'g_train_loss')
plt.plot(data['c_0_train_loss'], label = 'client_1_train_loss')
plt.plot(data['c_1_train_loss'], label = 'client_2_train_loss')
plt.plot(data['c_2_train_loss'], label = 'client_3_train_loss')
plt.plot(data['c_3_train_loss'], label = 'client_4_train_loss')
plt.xticks(np.arange(0,5,1))
plt.ylim(0,15)
plt.legend()
plt.show()

In [ ]:
plt.plot(data['g_test_loss'], label = 'g_test_loss')
plt.plot(data['c_0_test_loss'], label = 'client_0_test_loss')
plt.plot(data['c_1_test_loss'], label = 'client_1_test_loss')
plt.plot(data['c_2_test_loss'], label = 'client_2_test_loss')
plt.plot(data['c_3_test_loss'], label = 'client_3_test_loss')
plt.xticks(np.arange(0,5,1))
plt.ylim(0,15)
plt.legend()
plt.show()

In [ ]:
print(global_model)
print(global_model.state_dict())

---
### Evaluating the global model on combined test data vs individual datas

In [ ]:
criterion = nn.CrossEntropyLoss()
def post_trained_global_model(model, test_loader, criterion, device):
    model.eval()
    test_loss = 0.0
    total = 0 
    correct = 0 
    true_labels = []
    prediction = []

    with torch.no_grad():
        for samples, features in test_loader:
            samples = samples.to(device)
            features = features.to(device)
            output = model(samples)
            loss = criterion(output, features)
            _, predicted = output.max(1)
            prediction.extend(predicted.tolist())
            total += features.size(0)
            test_loss += loss.item()
            correct += predicted.eq(features).sum().item()
            true_labels.extend(features.tolist())

        test_loss = test_loss/len(test_loader.dataset)
        accuracy = 100* correct / total 
    
    return test_loss, accuracy, prediction, true_labels

In [ ]:
global_dataset.head(5)

### Creating the global dataset loader

In [ ]:
## Converting the global tumple to the dataloader
scaler = StandardScaler()
X = global_dataset.drop(columns ='Label_Binary')
X = X.to_numpy()
y = global_dataset['Label_Binary']
y = y.to_numpy()
X_tensor = torch.tensor(X, dtype = torch.float32)
y_tensor = torch.tensor(y, dtype = torch.long)
global_tuple = TensorDataset(X_tensor,y_tensor)
test_loader = DataLoader(global_tuple, batch_size=batch_size, shuffle=True,generator=g)
global_test_loss, global_accuracy, prediction, true_labels = post_trained_global_model(global_model, test_loader,criterion,device)

In [ ]:
print(f"The test loss: {global_test_loss}")
print(f"The accuracy :{global_accuracy}")


In [ ]:
### Classification matrix
report = classification_report(prediction, true_labels)
print(report)

In [ ]:
centralized_model = utils.loading_pickle('../models/centralized_model.pickle')
centralized_model = centralized_model.to(device)
centralized_test_loss,centralized_accuracy,centralized_predictions,centralized_true_labels = post_trained_global_model(centralized_model, test_loader, criterion, device)
print(centralized_test_loss)
print(centralized_accuracy)
print(classification_report(centralized_predictions,centralized_true_labels))

### Analyzing the post-trained global model 




In [ ]:
def batch_maker(dataset):
    dataset = dataset.drop(columns = 'Unnamed: 0', errors='ignore')
    X = dataset.drop(columns = 'Label_Binary')
    X = X.to_numpy()
    y = dataset['Label_Binary']
    y = y.to_numpy()
    X_train , X_test, y_train, y_test = train_test_split(X,y , test_size=0.3, random_state=42)
    X_train = scaler.fit_transform(X_train)
    X_train = torch.tensor(X_train, dtype = torch.float32)
    y_train = torch.tensor(y_train, dtype = torch.long)

    X_test = scaler.transform(X_test)
    X_test = torch.tensor(X_test, dtype = torch.float32)
    y_test = torch.tensor(y_test, dtype = torch.long)

    training_batch = DataLoader(TensorDataset(X_train,y_train), batch_size = 64, shuffle = True,generator=g)
    testing_batch = DataLoader(TensorDataset(X_test,y_test), batch_size=64, shuffle=False,generator=g)
    
    return training_batch,testing_batch 

In [ ]:
siloOne_train, siloOne_test = batch_maker(pd.read_csv('../silos_datasets/siloBinaryOne.csv'))
siloTwo_train, siloTwo_test = batch_maker(pd.read_csv('../silos_datasets/siloBinaryTwo.csv'))
siloThree_train, siloThree_test= batch_maker(pd.read_csv('../silos_datasets/siloBinaryThree.csv'))
siloFour_train, siloFour_test = batch_maker(pd.read_csv('../silos_datasets/siloBinaryFour.csv'))


In [ ]:
test_loss, test_accuracy, predictions, true_labels = post_trained_global_model(global_model, siloOne_test, criterion, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(true_labels, predictions)
print(report)

In [ ]:
test_loss, test_accuracy, predictions, true_labels = post_trained_global_model(global_model, siloTwo_test, criterion, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(true_labels, predictions)
print(report)

In [ ]:
test_loss, test_accuracy, predictions, true_labels = post_trained_global_model(global_model, siloThree_test, criterion, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(true_labels, predictions)
print(report)

In [ ]:
test_loss, test_accuracy, predictions, true_labels = post_trained_global_model(global_model, siloFour_test, criterion, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(true_labels, predictions)
print(report)

In [ ]:
test_loader